In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

# Image Filtering and PyTorch Classification Models
This notebook demonstrates:
- Basic image filtering using scikit-image.
- A fully connected MNIST classifier.
- A LeNet5 MNIST classifier with Adam optimizer, weight decay, and a learning rate scheduler.
- A LeNet5 FashionMNIST classifier with the same training scheme.

## Initialize Devise

In [ ]:
def init_device():
    # For the most part I'll try to import functions and classes near
    # where they are used
    # to make it clear where they come from.
    return torch.device(
        'cuda' if torch.cuda.is_available() else (
            'mps' if torch.backends.mps.is_available() else 'cpu'
        )
    )

In [ ]:
device = init_device()
device

## General Imports

In [ ]:
from tqdm import tqdm

## Section 1: Image Filtering with scikit-image
We will load an image, apply a Sobel filter (for edge detection), and display the original and filtered images.

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
from skimage import data, filters, color

# Load a sample image from scikit-image (e.g., camera image)
image = data.camera()  # grayscale image

# Apply Sobel filter to detect edges
edges = filters.sobel(image)

# Plot the original image and the edge-detected image
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(image, cmap='gray')
ax[0].set_title('Original Image')
ax[0].axis('off')

ax[1].imshow(edges, cmap='gray')
ax[1].set_title('Sobel Filtered Image')
ax[1].axis('off')

plt.show()

## Section 2: Fully Connected MNIST Classifier (PyTorch)
In this section, we load MNIST, compute the mean and standard deviation on the training set, build a simple fully connected network, and train it.

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split

# Define transformation: convert to tensor only for now
transform = transforms.ToTensor()

# Download MNIST training dataset
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Calculate mean and std on training data
train_loader = DataLoader(mnist_train, batch_size=60000, shuffle=False)
train_data, _ = next(iter(train_loader))
print(f"Training Data Mean: {mean:.4f}, Std: {std:.4f}")

# Define transformation with normalization
transform_norm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

# Reload dataset with normalization
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_norm)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_norm)

# Split train into train and validation
train_size = int(0.8 * len(mnist_train))
val_size = len(mnist_train) - train_size
mnist_train, mnist_val = random_split(mnist_train, [train_size, val_size])

# Data loaders
train_loader = DataLoader(mnist_train, batch_size=64, shuffle=True)
val_loader   = DataLoader(mnist_val, batch_size=64, shuffle=False)
test_loader  = DataLoader(mnist_test, batch_size=64, shuffle=False)

mean = train_data.mean().item()
std = train_data.std().item()

# Define a simple fully connected network for MNIST
class FcMnist(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        
    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model_fc = FcMnist()
model_fc = model_fc.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_fc.parameters(), lr=0.001)

# Training loop for fully connected model
for epoch in range(5):  # use 5 epochs for demonstration
    model_fc.train()
    running_loss = 0.0
    with(tqdm(train_loader)) as ploader:
        for images, labels in ploader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model_fc(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
len(test_loader.dataset)

## Section 3: LeNet5 MNIST Classifier (PyTorch)
Now we will build a LeNet5-style network for MNIST. The model will use the Adam optimizer with weight decay and a learning rate scheduler. We will split the MNIST dataset into training, validation, and test sets.

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Define LeNet5-style network for MNIST
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5)  # 1 input channel, 6 output channels, 5x5 kernel
        self.pool = nn.AvgPool2d(2, 2)  # average pooling 2x2
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.fc1 = nn.Linear(16*4*4, 120)  # Adjusted for MNIST input size 28x28
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))      # conv1 + ReLU
        x = self.pool(x)               # pooling
        x = F.relu(self.conv2(x))      # conv2 + ReLU
        x = self.pool(x)               # pooling
        x = x.view(-1, 16*4*4)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        
        return x

lenet_5 = LeNet5()
lenet_5 = lenet_5.to(device)

In [ ]:
# Use Adam with weight decay
optimizer = optim.Adam(lenet_5.parameters(), lr=0.001, weight_decay=1e-4)
# Learning rate scheduler: step down LR by factor 0.1 every 10 epochs
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
criterion = nn.CrossEntropyLoss()

# Training loop for LeNet5 MNIST classifier
num_epochs = 15
for epoch in range(num_epochs):
    lenet_5.train()
    running_loss = 0.0
    with(tqdm(train_loader)) as ptrain:
        for images, labels in ptrain:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = lenet_5(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")
    
# Evaluate on test data
lenet_5.eval()
correct = 0
total = 0
with torch.no_grad():
    with(tqdm(test_loader)) as ptest:
        for images, labels in ptest:
            images = images.to(device)
            labels = labels.to(device)
            outputs = lenet_5(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

In [ ]:
test_loader.dataset[0][0].shape

## Section 4: LeNet5 FashionMNIST Classifier (PyTorch)
Similarly, we build a LeNet5-style network for FashionMNIST. The process is analogous to the MNIST classifier.

In [ ]:
# Import FashionMNIST from torchvision
import torchvision.datasets as datasets

# Define transformations (using normalization)
fashion_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Download FashionMNIST dataset
fashion_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=fashion_transform)
fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=fashion_transform)

# Split fashion_train into train and validation sets
train_size = int(0.8 * len(fashion_train))
val_size = len(fashion_train) - train_size
fashion_train, fashion_val = random_split(fashion_train, [train_size, val_size])

fashion_train_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=64, shuffle=False)
fashion_test_loader  = DataLoader(fashion_test, batch_size=64, shuffle=False)

# Calculate mean and std on training data
fashion_train_loader_mean = DataLoader(fashion_train, batch_size=60000, shuffle=False)
fashion_train_data, _ = next(iter(fashion_train_loader))
mean = fashion_train_data.mean().item()
std = fashion_train_data.std().item()
print(f"Training Data Mean: {mean:.4f}, Std: {std:.4f}")

# Define transformations (using normalization)
fashion_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))  # using a simple normalization for FashionMNIST
])

# Download FashionMNIST dataset
fashion_train = datasets.FashionMNIST(root='./data', train=True, download=True, transform=fashion_transform)
fashion_test = datasets.FashionMNIST(root='./data', train=False, download=True, transform=fashion_transform)

# Split fashion_train into train and validation sets
train_size = int(0.8 * len(fashion_train))
val_size = len(fashion_train) - train_size
fashion_train, fashion_val = random_split(fashion_train, [train_size, val_size])

fashion_train_loader = DataLoader(fashion_train, batch_size=64, shuffle=True)
fashion_val_loader   = DataLoader(fashion_val, batch_size=64, shuffle=False)
fashion_test_loader  = DataLoader(fashion_test, batch_size=64, shuffle=False)

# Define LeNet5 model for FashionMNIST (reuse same architecture as for MNIST)
lenet_fashion = LeNet5()  # LeNet5_MNIST defined earlier
lenet_fashion = lenet_fashion.to(device)

In [ ]:
optimizer = optim.Adam(lenet_fashion.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
criterion = nn.CrossEntropyLoss()

# Training loop for LeNet5 FashionMNIST classifier
num_epochs = 15
for epoch in range(num_epochs):
    lenet_fashion.train()
    running_loss = 0.0
    with(tqdm(fashion_train_loader)) as ptrain:
        for images, labels in ptrain:
            optimizer.zero_grad()
            images = images.to(device)
            labels = labels.to(device)
            outputs = lenet_fashion(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(fashion_train_loader):.4f}")
    
# Evaluate on test data
lenet_fashion.eval()
correct = 0
total = 0
with torch.no_grad():
    with(tqdm(fashion_test_loader)) as ptest:
        for images, labels in ptest:
            images = images.to(device)
            labels = labels.to(device)
            outputs = lenet_fashion(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
print(f"FashionMNIST Test Accuracy: {100 * correct / total:.2f}%")

Final Notes
	-Image Filtering:
The notebook uses a sample image from scikit-image. You may change the image source if desired.
	-Datasets:
The MNIST and FashionMNIST datasets are automatically downloaded via torchvision. Data is split into training, validation, and test sets.
	-Model Architectures:
A simple fully connected network is provided for MNIST, and a LeNet5-style model is used for both MNIST and FashionMNIST classifiers.
	-	Training Details:
The models are trained with the Adam optimizer (with weight decay), and a learning rate scheduler (StepLR) is used.

Feel free to adjust the number of epochs, batch sizes, and other hyperparameters as needed for your teaching purposes.